In [2]:
# Project 2 – Data Cleaning, Missing Data, Transformations

In [3]:
# 1. Load the dataset

In [4]:
import pandas as pd 
import numpy as np

In [5]:

df = pd.read_csv("/Users/keerthanat/Desktop/marketing_campaign_dataset.csv")

In [6]:
# Display basic information to understand the structure of the data
df.shape
df.head()
df.columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Campaign_ID       200000 non-null  int64  
 1   Company           200000 non-null  object 
 2   Campaign_Type     200000 non-null  object 
 3   Target_Audience   200000 non-null  object 
 4   Duration          200000 non-null  object 
 5   Channel_Used      200000 non-null  object 
 6   Conversion_Rate   200000 non-null  float64
 7   Acquisition_Cost  200000 non-null  object 
 8   ROI               200000 non-null  float64
 9   Location          200000 non-null  object 
 10  Language          200000 non-null  object 
 11  Clicks            200000 non-null  int64  
 12  Impressions       200000 non-null  int64  
 13  Engagement_Score  200000 non-null  int64  
 14  Customer_Segment  200000 non-null  object 
 15  Date              200000 non-null  object 
dtypes: float64(2), int64

In [7]:
# 2) Keep only columns relevant to the research question
cols_used = [
    "Campaign_ID", "Company", "Campaign_Type", "Target_Audience",
    "Duration", "Channel_Used",
    "Conversion_Rate", "Acquisition_Cost", "ROI",
    "Clicks", "Impressions", "Engagement_Score",
    "Date"
]
df = df[cols_used].copy()

In [8]:
# Confirm we kept the correct columns
df.shape
df.columns

Index(['Campaign_ID', 'Company', 'Campaign_Type', 'Target_Audience',
       'Duration', 'Channel_Used', 'Conversion_Rate', 'Acquisition_Cost',
       'ROI', 'Clicks', 'Impressions', 'Engagement_Score', 'Date'],
      dtype='object')

In [9]:
# 3) Check missing values (even if df.info() shows non-null)
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_counts


Campaign_ID         0
Company             0
Campaign_Type       0
Target_Audience     0
Duration            0
Channel_Used        0
Conversion_Rate     0
Acquisition_Cost    0
ROI                 0
Clicks              0
Impressions         0
Engagement_Score    0
Date                0
dtype: int64

In [10]:
# Percent missing values per column 
missing_perc = (df.isna().mean() * 100).sort_values(ascending=False)
missing_perc

Campaign_ID         0.0
Company             0.0
Campaign_Type       0.0
Target_Audience     0.0
Duration            0.0
Channel_Used        0.0
Conversion_Rate     0.0
Acquisition_Cost    0.0
ROI                 0.0
Clicks              0.0
Impressions         0.0
Engagement_Score    0.0
Date                0.0
dtype: float64

In [11]:
# 4) Fix data types
# Duration is currently an object; convert to numeric so we can analyze it.
df["Duration"] = (
    df["Duration"]
    .astype(str)                          # make sure it's string
    .str.extract(r"(\d+)", expand=False)  # extract the number part
)

# Convert extracted values to numeric
df["Duration"] = pd.to_numeric(df["Duration"], errors="coerce")

# (Optional) Fill any missing values created by bad strings, then convert to int
df["Duration"] = df["Duration"].fillna(df["Duration"].median()).astype(int)
# Acquisition_Cost is currently an object (may include $ signs or commas).
df["Acquisition_Cost"] = (
    df["Acquisition_Cost"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)
df["Acquisition_Cost"] = pd.to_numeric(df["Acquisition_Cost"], errors="coerce")
# Convert Date from object to datetime so we can do time-based analysis later if needed.
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
# Convert categorical variables to category type
cat_cols = ["Company", "Campaign_Type", "Target_Audience", "Channel_Used"]
for c in cat_cols:
    df[c] = df[c].astype("category")


In [12]:
df.dtypes

Campaign_ID                  int64
Company                   category
Campaign_Type             category
Target_Audience           category
Duration                     int64
Channel_Used              category
Conversion_Rate            float64
Acquisition_Cost           float64
ROI                        float64
Clicks                       int64
Impressions                  int64
Engagement_Score             int64
Date                datetime64[ns]
dtype: object

In [13]:
df["Duration"].head(10)

0    30
1    60
2    30
3    60
4    15
5    15
6    60
7    45
8    15
9    15
Name: Duration, dtype: int64

In [14]:
# 5) Handle missing values created by type conversion (if any)
df.isna().sum().sort_values(ascending=False)

Campaign_ID         0
Company             0
Campaign_Type       0
Target_Audience     0
Duration            0
Channel_Used        0
Conversion_Rate     0
Acquisition_Cost    0
ROI                 0
Clicks              0
Impressions         0
Engagement_Score    0
Date                0
dtype: int64

In [15]:
# For Acquisition_Cost: if missing values exist, impute with median
df["Acquisition_Cost"] = df["Acquisition_Cost"].fillna(df["Acquisition_Cost"].median())
# Drop rows missing key predictors/outcomes needed for analysis
df = df.dropna(subset=["Channel_Used", "Target_Audience", "Duration", "Conversion_Rate", "ROI"])

In [16]:
# 6) Remove invalid/impossible values (data cleaning)
df = df[df["Duration"] > 0]
df = df[(df["Clicks"] >= 0) & (df["Impressions"] >= 0) & (df["Engagement_Score"] >= 0)]
df = df[df["Acquisition_Cost"] >= 0]

In [17]:
df.shape

(200000, 13)

In [18]:
# 7) Data transformations (new variables that help analysis)
#Cost per click (CPC) as an additional efficiency metric
df["CPC"] = np.where(df["Clicks"] > 0, df["Acquisition_Cost"] / df["Clicks"], np.nan)
#Click-through rate (CTR)
df["CTR"] = np.where(df["Impressions"] > 0, df["Clicks"] / df["Impressions"], np.nan)
#Bin duration into categories
df["Duration_cat"] = pd.cut(
    df["Duration"],
    bins=[-np.inf, 7, 30, np.inf],
    labels=["Short", "Medium", "Long"]
)

In [19]:
# 8) Create dummy variables for categorical predictors (Channel_Used, Target_Audience)
dummies = pd.get_dummies(df[["Channel_Used", "Target_Audience"]], drop_first=True)
df = pd.concat([df, dummies], axis=1)


In [20]:
df.shape
df.head()
df.isna().sum().sort_values(ascending=False)
df.describe(include="all")


,Campaign_ID,Company,Campaign_Type,Target_Audience,Duration,Channel_Used,Conversion_Rate,Acquisition_Cost,ROI,Clicks,...,Duration_cat,Channel_Used_Facebook,Channel_Used_Google Ads,Channel_Used_Instagram,Channel_Used_Website,Channel_Used_YouTube,Target_Audience_Men 18-24,Target_Audience_Men 25-34,Target_Audience_Women 25-34,Target_Audience_Women 35-44
count,200000.000000,200000,200000,200000,200000.000000,200000,200000.000000,200000.000000,200000.000000,200000.000000,...,200000,200000,200000,200000,200000,200000,200000,200000,200000,200000
unique,NaN,5,5,5,NaN,6,NaN,NaN,NaN,NaN,...,2,2,2,2,2,2,2,2,2,2
top,NaN,TechCorp,Influencer,Men 18-24,NaN,Email,NaN,NaN,NaN,NaN,...,Medium,False,False,False,False,False,False,False,False,False
freq,NaN,40237,40169,40258,NaN,33599,NaN,NaN,NaN,NaN,...,100034,167181,166562,166608,166640,166608,159742,159977,159987,160313
mean,100000.500000,NaN,NaN,NaN,37.503975,NaN,0.080070,12504.393040,5.002438,549.772030,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,1.000000,NaN,NaN,NaN,15.000000,NaN,0.010000,5000.000000,2.000000,100.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,50000.750000,NaN,NaN,NaN,30.000000,NaN,0.050000,8739.750000,3.500000,325.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,100000.500000,NaN,NaN,NaN,30.000000,NaN,0.080000,12496.500000,5.010000,550.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,150000.250000,NaN,NaN,NaN,45.000000,NaN,0.120000,16264.000000,6.510000,775.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,200000.000000,NaN,NaN,NaN,60.000000,NaN,0.150000,20000.000000,8.000000,1000.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
# Project 3 Head: 
#1)Processing Strings
#2) Data Wrangling-The dataset consists of a single CSV file. Therefore, no merging or combining of multiple datasets was required.
#3)Data Wrangling: Reshaping and Pivoting- dimensions: Channel, Campaign Type, Target Audience, and Duration.

In [24]:
# Goal: clean and extract useful information from text columns
#1a) Standardize string columns: strip whitespace and apply consistent casing
str_cols = ['Company', 'Campaign_Type', 'Target_Audience', 'Channel_Used']
for col in str_cols:
    df[col] = df[col].astype('str').str.strip().str.title()

print('Unique Channel_Used values:', df['Channel_Used'].unique().tolist())
print('Unique Campaign_Type values:', df['Campaign_Type'].unique().tolist())
print('Unique Target_Audience values:', df['Target_Audience'].unique().tolist())


Unique Channel_Used values: ['Google Ads', 'Youtube', 'Instagram', 'Website', 'Facebook', 'Email']
Unique Campaign_Type values: ['Email', 'Influencer', 'Display', 'Search', 'Social Media']
Unique Target_Audience values: ['Men 18-24', 'Women 35-44', 'Men 25-34', 'All Ages', 'Women 25-34']


In [25]:
# 1b) Extract Gender and Age_Group from Target_Audience using regex.
# Target_Audience values look like 'Men 18-24', 'Women 25-34', etc.
# Splitting into two columns makes group-level analysis easier.

df['Gender']    = df['Target_Audience'].str.extract(r'^(\w+)')
df['Age_Group'] = df['Target_Audience'].str.extract(r'(\d{2}-\d{2})')

print(df[['Target_Audience', 'Gender', 'Age_Group']].drop_duplicates().sort_values('Target_Audience'))

   Target_Audience Gender Age_Group
3         All Ages    All       NaN
0        Men 18-24    Men     18-24
2        Men 25-34    Men     25-34
20     Women 25-34  Women     25-34
1      Women 35-44  Women     35-44


In [26]:
# 1c) Build a Campaign_Label string by combining Campaign_Type and Channel_Used.
# This creates a concise identifier useful for grouping and chart labels.

df['Campaign_Label'] = (
    df['Campaign_Type'].str.replace(' ', '', regex=False)
    + '_'
    + df['Channel_Used'].str.replace(' ', '', regex=False)
)
print('Top 10 Campaign Labels by count:')
print(df['Campaign_Label'].value_counts().head(10))

Top 10 Campaign Labels by count:
Campaign_Label
Search_Website          6899
Influencer_Youtube      6773
Influencer_GoogleAds    6764
Email_Email             6760
Influencer_Instagram    6736
Display_Instagram       6724
Display_Youtube         6722
SocialMedia_Email       6722
Search_GoogleAds        6720
Search_Email            6718
Name: count, dtype: int64


In [28]:
# 1d) Extract Month-Year as a string period label from the Date column.
# Date was already parsed to datetime in Part 2, so this is straightforward.

df['Month_Year'] = df['Date'].dt.to_period('M').astype('str')  # e.g. '2021-03'
print('Sample Month_Year values (sorted):')
print(df['Month_Year'].value_counts().sort_index().head(6))

Sample Month_Year values (sorted):
Month_Year
2021-01    16988
2021-02    15344
2021-03    16988
2021-04    16440
2021-05    16988
2021-06    16440
Name: count, dtype: int64


In [ ]:
#2) Data Wrangling-The dataset consists of a single CSV file.

In [36]:
# 3a) Pivot 1: Average ROI by Channel and Campaign Type
roi_pivot = df.pivot_table(
    values  = 'ROI',
    index   = 'Channel_Used',
    columns = 'Campaign_Type',
    aggfunc = 'mean'
).round(3)

print('Average ROI by Channel and Campaign Type:')
print(roi_pivot)

Average ROI by Channel and Campaign Type:
Campaign_Type  Display  Email  Influencer  Search  Social Media
Channel_Used                                                   
Email            5.031  5.003       4.991   4.973         4.985
Facebook         5.001  5.012       5.027   5.033         5.022
Google Ads       5.003  5.011       5.002   5.005         4.994
Instagram        4.970  4.991       4.998   5.007         4.978
Website          5.031  4.991       5.025   5.032         4.991
Youtube          5.003  4.957       5.024   5.000         4.983


In [37]:
# 3b) Pivot 2: Average Conversion Rate by Channel and Target Audience
conv_pivot = df.pivot_table(
    values  = 'Conversion_Rate',
    index   = 'Channel_Used',
    columns = 'Target_Audience',
    aggfunc = 'mean'
).round(4)

print('Average Conversion Rate by Channel and Target Audience:')
print(conv_pivot)

Average Conversion Rate by Channel and Target Audience:
Target_Audience  All Ages  Men 18-24  Men 25-34  Women 25-34  Women 35-44
Channel_Used                                                             
Email              0.0793     0.0807     0.0805       0.0799       0.0810
Facebook           0.0795     0.0805     0.0801       0.0797       0.0800
Google Ads         0.0805     0.0801     0.0807       0.0797       0.0799
Instagram          0.0804     0.0798     0.0795       0.0801       0.0797
Website            0.0803     0.0805     0.0800       0.0801       0.0800
Youtube            0.0798     0.0799     0.0800       0.0798       0.0800


In [38]:
#3c) Pivot 3: Average ROI and count by Duration Category. 
pivot_roi_duration = df.pivot_table(
    values="ROI",
    index="Duration_cat",
    aggfunc="mean",
    observed=False 
)

pivot_roi_duration

,ROI
Duration_cat,
Medium,5.002832
Long,5.002043


In [39]:
# 3c) Reshaping with melt: convert the wide roi_pivot back to long (tidy) format.
# Long format is more flexible for plotting with seaborn / matplotlib.
roi_long = roi_pivot.reset_index().melt(
    id_vars   = 'Channel_Used',
    var_name  = 'Campaign_Type',
    value_name= 'Avg_ROI'
)

print('Long-format ROI table (first 10 rows):')
print(roi_long.head(10))
print(f'\nTotal rows in long format: {len(roi_long)}')

Long-format ROI table (first 10 rows):
  Channel_Used Campaign_Type  Avg_ROI
0        Email       Display    5.031
1     Facebook       Display    5.001
2   Google Ads       Display    5.003
3    Instagram       Display    4.970
4      Website       Display    5.031
5      Youtube       Display    5.003
6        Email         Email    5.003
7     Facebook         Email    5.012
8   Google Ads         Email    5.011
9    Instagram         Email    4.991

Total rows in long format: 30
